# Pregunta 1 — Entrenamiento de los modelos A y B en Colab

Requisitos: `colab_bundle.zip` (generado con `python src/s07_pack_colab.py`) subido a
`MyDrive/knee/` en una carpeta **privada** (datos derivados de Kaggle).

Entorno: *Entorno de ejecución → Cambiar tipo → GPU* (L4 o A100 en Colab Pro).
Cada corrida guarda `last.pt` en Drive al final de cada época: si Colab se desconecta,
basta volver a ejecutar la celda de corridas (usa `--resume`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/knee'
WORK = '/content/knee'
# Copia al disco local de Colab: leer desde Drive archivo por archivo es lento
!mkdir -p {WORK} && cp {DRIVE}/colab_bundle.zip /content/ && unzip -q -o /content/colab_bundle.zip -d {WORK}
%cd {WORK}/src
!ls ../data/processed

## Corridas

| Nombre | Qué cambia frente a `B_base` | Entregable |
|---|---|---|
| `A_base` | CNN desde cero (modelo A) | 2 |
| `B_base` | ResNet-50 ImageNet, sagital, BCE + pos_weight, etiquetas duras | 2, 3 |
| `B_focal` | pérdida focal (γ = 2) | 4 (pérdida) |
| `B_cor` | plano coronal | 4 (plano) |
| `B_conf` | pondera cada etiqueta por su confianza | 4 (ruido) |
| `B_hc` | solo etiquetas de alta confianza | 4 (ruido) |
| `B_nopw` | sin `pos_weight` (sin mitigación del desbalance) | 6 |

Para repartir entre integrantes, cada uno deja en `RUN` solo sus corridas.

In [ ]:
RUNS = {
    'A_base':  '--arch scratch',
    'B_base':  '--arch resnet50',
    'B_focal': '--arch resnet50 --loss focal',
    'B_cor':   '--arch resnet50 --view cor',
    'B_conf':  '--arch resnet50 --noise conf',
    'B_hc':    '--arch resnet50 --noise hc',
    'B_nopw':  '--arch resnet50 --no-pos-weight',
}
RUN = ['B_base', 'A_base']   # orden de prioridad; luego las ablaciones

for name in RUN:
    print(f'
==================== {name}')
    !python train.py {RUNS[name]} --workers 4 --resume --out {DRIVE}/runs/{name}

In [ ]:
import pandas as pd, glob, os
rows = []
for d in sorted(glob.glob(f'{DRIVE}/runs/*/history.csv')):
    h = pd.read_csv(d)
    rows.append(dict(run=os.path.basename(os.path.dirname(d)), epochs=len(h),
                     best_auc_weak_val=h.val_auc_weak.max(), min_per_epoch=h.sec.mean() / 60))
pd.DataFrame(rows).round(3)